# 08-4. 안전한 외부 프로세스 실행 예제

## Goal

- 명령 문자열 대신 인자 목록을 사용합니다.
- 종료 코드·출력 크기·타임아웃을 확인합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

현재 Python을 자식 프로세스로 실행하며 셸과 외부 프로그램은 사용하지 않습니다.


## Steps

### 안전한 subprocess 경계

고정된 실행 파일과 인자 목록을 사용하고 출력과 시간에 제한을 둡니다.


In [1]:
import subprocess
import sys


def run_python(code: str, timeout=2.0, output_limit=4096):
    completed = subprocess.run(
        [sys.executable, "-c", code],
        shell=False,
        capture_output=True,
        text=True,
        timeout=timeout,
        check=False,
    )
    if len(completed.stdout) + len(completed.stderr) > output_limit:
        raise ValueError("자식 프로세스 출력 제한을 초과했습니다")
    return {"returncode": completed.returncode, "stdout": completed.stdout, "stderr": completed.stderr}


process_result = run_python("print('child-ok')")
print(process_result)


{'returncode': 0, 'stdout': 'child-ok\n', 'stderr': ''}


## Checks

정상 종료와 비정상 종료를 구분합니다.


In [2]:
assert process_result == {"returncode": 0, "stdout": "child-ok\n", "stderr": ""}
failed = run_python("import sys; sys.exit(7)")
assert failed["returncode"] == 7
print("프로세스 종료 상태 검사 통과")


프로세스 종료 상태 검사 통과


## Next Steps

사용자가 입력한 문자열을 `shell=True`로 전달하지 않으며 실행 파일 허용 목록을 별도로 관리합니다.
